In [ ]:
#GitHub CI - Search from Stard Boundary commit to Cut-off commit for Instru tests performance metrics
#but will keep it high level for perventing 90 days log limit
# it does check_run & commit Status (external CI services data) and more

#up to 100 samples per episode that are observable through GitHub check runs or commit statuses and match the instrumentation regex, within scan limits

In [2]:
import csv
import json
import os
import re
import time
import random
from dataclasses import dataclass
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Union, Tuple

import requests
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
ROOT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_B")

TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

URL_LIST_CSV = ROOT / "URL_List_Instru.csv"
EPISODES_CSV  = ROOT / "List_Change_Episodes.csv"

OUT_DIR = ROOT / "RouteA_EpisodeRunMetrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_OUT = OUT_DIR / "routeA_runs_labeled.csv"
WF_EP_OUT = OUT_DIR / "routeA_workflow_episode_metrics.csv"
STYLE_OUT = OUT_DIR / "routeA_style_overall_metrics.csv"

CHECKPOINT_PATH = OUT_DIR / "routeA_checkpoint.json"

# Hard cutoff (inclusive end-of-day UTC)
CUTOFF_UTC = datetime(2025, 8, 10, 23, 59, 59, tzinfo=timezone.utc)

# Resume automatically if checkpoint exists
AUTO_RESUME = True

# Stop condition: collect K top-level instrumentation CI *matches* per episode
# NOTE: Steps can add extra rows; K only controls “top-level” matches (check_run + commit_status).
K_INSTRU_RECORDS_PER_EPISODE = 100

INITIAL_WINDOW_DAYS = 30
MAX_WINDOW_DAYS = 3650
MAX_COMMITS_TO_SCAN_PER_EPISODE = 5000

PROVIDER_FOCUS = "both"  # gha_only | non_gha_only | both

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

# ============================================================
# Instrumentation-ish matching
# ============================================================
INSTRU_NAME_RE = re.compile(
    r"(androidtest|connectedandroidtest|connected.*android.*test|connectedcheck|devicecheck|"
    r"manageddevice|gmd|instrument(ed|ation)?|am\s+instrument|espresso|uiautomator|ui\s*test|"
    r"firebase\s+test|test\s*lab|device\s*farm|browserstack|sauce(labs)?|kobiton|appcenter|"
    r"flank|maestro|marathon|spoon|baselineprofile|benchmark)",
    re.IGNORECASE,
)

def is_instru(s: str) -> bool:
    return bool(INSTRU_NAME_RE.search(s or ""))

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def seconds_between(a: Optional[str], b: Optional[str]) -> Optional[int]:
    try:
        da = pd.to_datetime(a, utc=True)
        db = pd.to_datetime(b, utc=True)
        sec = int((db - da).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None

# ============================================================
# Output schema (fixed header!)
# ============================================================
FIELDNAMES = [
    "full_name","default_branch",
    "episode_id","episode_env_styles","episode_start_utc","episode_end_utc",
    "commit_sha",

    # general record identity
    "record_type","level",
    "provider","provider_kind","is_github_actions",

    # workflow run
    "workflow_run_id","workflow_run_attempt","workflow_name","workflow_event",
    "head_branch","head_sha","run_created_at","run_updated_at",

    # job
    "job_id","job_name","job_status","job_conclusion","runner_name","runner_labels",

    # step
    "step_number","step_name","step_status","step_conclusion",

    # timings (common fields)
    "started_at","completed_at","duration_seconds",

    # urls + bookkeeping
    "html_url","details_url",
    "collected_at_utc","run_instance_key",
]

def _row_with_defaults(d: Dict) -> Dict:
    out = {k: "" for k in FIELDNAMES}
    for k, v in d.items():
        if k in out:
            out[k] = v
    return out

def flush_rows(rows: List[Dict], out_path: Path):
    if not rows:
        return
    write_header = not out_path.exists()
    with out_path.open("a", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=FIELDNAMES, extrasaction="ignore")
        if write_header:
            w.writeheader()
        for r in rows:
            w.writerow(_row_with_defaults(r))

# ============================================================
# Resume / checkpoint
# ============================================================
def load_checkpoint() -> Dict:
    if not CHECKPOINT_PATH.exists():
        return {"version": 1, "cutoff_utc": CUTOFF_UTC.isoformat(), "episode_state": {}, "episode_done": {}}
    try:
        return json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
    except Exception:
        return {"version": 1, "cutoff_utc": CUTOFF_UTC.isoformat(), "episode_state": {}, "episode_done": {}}

def save_checkpoint(cp: Dict) -> None:
    cp["last_updated_utc"] = now_utc_iso()
    CHECKPOINT_PATH.write_text(json.dumps(cp, indent=2, sort_keys=True), encoding="utf-8")

def ep_key(full_name: str, episode_id: Union[str,int]) -> str:
    return f"{full_name}||{episode_id}"

# ============================================================
# Repo parsing helpers
# ============================================================
def parse_repo_full_name(repo_url_or_fullname: str) -> Optional[str]:
    s = (repo_url_or_fullname or "").strip()
    if not s:
        return None
    if re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", s):
        return s
    m = re.search(r"github\.com[:/]+([A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+)", s, re.IGNORECASE)
    if m:
        full = m.group(1)
        return full[:-4] if full.endswith(".git") else full
    return None

def allow_check_run(app_slug: str) -> bool:
    slug = (app_slug or "").strip().lower()
    if PROVIDER_FOCUS == "both":
        return True
    if PROVIDER_FOCUS == "gha_only":
        return slug == "github-actions"
    if PROVIDER_FOCUS == "non_gha_only":
        return slug != "github-actions"
    return True

def allow_commit_status() -> bool:
    return PROVIDER_FOCUS != "gha_only"

# ============================================================
# GitHub client
# ============================================================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "episode-ci-metrics-k/2.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        time.sleep(max(1, soonest - now + 2))

    def _backoff(self, attempt: int) -> None:
        time.sleep(min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random())

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code
            try:
                st.remaining = int(resp.headers.get("X-RateLimit-Remaining", "0"))
            except Exception:
                pass
            try:
                st.reset_epoch = int(resp.headers.get("X-RateLimit-Reset", "0"))
            except Exception:
                pass

            if resp.status_code == 404:
                return None

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and ("rate limit" in text_l or "secondary rate limit" in text_l):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate_list(self, url: str, params: Optional[Dict] = None) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if not data or not isinstance(data, list):
                return
            if not data:
                return
            for it in data:
                yield it
            if len(data) < 100:
                return
            page += 1

# ============================================================
# GitHub endpoints (existing)
# ============================================================
def get_repo_default_branch(gh: GitHubClient, full_name: str) -> str:
    url = f"https://api.github.com/repos/{full_name}"
    data = gh.request_json("GET", url)
    if not data or not isinstance(data, dict):
        return ""
    return (data.get("default_branch") or "").strip()

def list_commits_in_range(gh: GitHubClient, full_name: str, branch: str, since_iso: str, until_iso: str) -> List[str]:
    url = f"https://api.github.com/repos/{full_name}/commits"
    params = {"sha": branch, "since": since_iso, "until": until_iso}
    shas: List[str] = []
    for c in gh.paginate_list(url, params=params):
        sha = c.get("sha")
        if isinstance(sha, str) and len(sha) >= 7:
            shas.append(sha)
    return shas  # newest -> oldest

def list_check_runs_for_commit(gh: GitHubClient, full_name: str, sha: str) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/commits/{sha}/check-runs"
    data = gh.request_json("GET", url)
    if not data or not isinstance(data, dict):
        return []
    return data.get("check_runs", []) or []

def get_combined_status_for_commit(gh: GitHubClient, full_name: str, sha: str) -> Dict:
    url = f"https://api.github.com/repos/{full_name}/commits/{sha}/status"
    data = gh.request_json("GET", url)
    return data if isinstance(data, dict) else {}

# ============================================================
# NEW: GitHub Actions run -> jobs -> steps (tied to check_suite_id)
# ============================================================
def get_workflow_run_by_check_suite_id(gh: GitHubClient, full_name: str, check_suite_id: int) -> Optional[Dict]:
    # This endpoint supports filtering by check_suite_id. :contentReference[oaicite:1]{index=1}
    url = f"https://api.github.com/repos/{full_name}/actions/runs"
    params = {"check_suite_id": check_suite_id, "per_page": 1, "page": 1}
    data = gh.request_json("GET", url, params=params)
    if not data or not isinstance(data, dict):
        return None
    runs = data.get("workflow_runs") or []
    return runs[0] if runs else None

def list_jobs_for_run(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    # Jobs contain steps (when available). :contentReference[oaicite:2]{index=2}
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    out: List[Dict] = []
    page = 1
    while page <= MAX_PAGES_PER_LIST:
        params = {"per_page": 100, "page": page}
        data = gh.request_json("GET", url, params=params)
        if not data or not isinstance(data, dict):
            break
        jobs = data.get("jobs") or []
        if not jobs:
            break
        out.extend(jobs)
        if len(jobs) < 100:
            break
        page += 1
    return out

# ============================================================
# Episodes loader (with cutoff)
# ============================================================
def build_repo_canon_map(repos: List[str]) -> Dict[str, str]:
    return {r.lower(): r for r in repos}

def normalize_episode_repo(row: pd.Series, repo_map: Dict[str, str]) -> Optional[str]:
    candidates: List[str] = []
    for key in ["repo_name", "full_name", "html_url", "url", "repo", "repository", "repo_url"]:
        if key in row and pd.notna(row[key]):
            s = str(row[key]).strip()
            if s:
                candidates.append(s)

    for raw in candidates:
        p = parse_repo_full_name(raw)
        if p and p.lower() in repo_map:
            return repo_map[p.lower()]

        if "__" in raw and raw.count("__") == 1:
            p2 = raw.replace("__", "/")
            if p2.lower() in repo_map:
                return repo_map[p2.lower()]

        if "/" not in raw and "." in raw:
            p3 = raw.replace(".", "/", 1)
            if p3.lower() in repo_map:
                return repo_map[p3.lower()]

    return None

def load_episodes(path: Path, repo_map: Dict[str, str]) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")

    if "episode_start_utc" not in df.columns or "episode_end_utc" not in df.columns:
        raise ValueError(f"Missing episode_start_utc/episode_end_utc in {path.name}. Columns={list(df.columns)}")

    out = df.copy()
    out["episode_start_utc"] = pd.to_datetime(out["episode_start_utc"], utc=True, errors="coerce")
    out["episode_end_utc"]   = pd.to_datetime(out["episode_end_utc"], utc=True, errors="coerce")

    style_col = "env_styles" if "env_styles" in out.columns else None
    out["episode_env_styles"] = out[style_col].astype(str) if style_col else ""

    if "episode_id" in out.columns:
        out["episode_id"] = out["episode_id"]
    elif "episode_index" in out.columns:
        out["episode_id"] = out["episode_index"]
    else:
        out["episode_id"] = range(len(out))

    out["full_name"] = out.apply(lambda r: normalize_episode_repo(r, repo_map), axis=1)
    out = out.dropna(subset=["full_name", "episode_start_utc", "episode_end_utc"])

    # apply cutoff: skip episodes fully after cutoff, and cap end at cutoff
    out = out[out["episode_start_utc"] <= CUTOFF_UTC].copy()
    out["episode_end_utc"] = out["episode_end_utc"].apply(lambda d: min(d, CUTOFF_UTC) if pd.notna(d) else d)
    out = out[out["episode_start_utc"] < out["episode_end_utc"]].copy()

    return out[["full_name", "episode_id", "episode_start_utc", "episode_end_utc", "episode_env_styles"]]

# ============================================================
# Aggregations (write repeatedly)
# ============================================================
def recompute_aggregates():
    if not RAW_OUT.exists():
        return
    df = pd.read_csv(RAW_OUT, encoding="utf-8-sig")
    if len(df) == 0:
        return

    # Only aggregate “top-level CI” rows (avoid counting steps as runs)
    top = df[df["record_type"].isin(["check_run","commit_status"])].copy()
    if len(top) == 0:
        return

    c = top["job_conclusion"].astype(str).str.lower()
    # backward compatibility: some rows store conclusion in job_conclusion
    top["is_success"] = c.isin({"success", "neutral", "skipped"})
    top["is_failure"] = c.isin({"failure", "error", "cancelled", "timed_out", "action_required"})
    top["duration_s"] = pd.to_numeric(top["duration_seconds"], errors="coerce")

    gcols = ["full_name", "episode_id", "episode_env_styles", "provider_kind", "provider", "job_name"]
    wf_ep = top.groupby(gcols).agg(
        runs=("run_instance_key", "nunique"),
        success_runs=("is_success", "sum"),
        failure_runs=("is_failure", "sum"),
        success_rate=("is_success", "mean"),
        dur_mean_s=("duration_s", "mean"),
        dur_median_s=("duration_s", "median"),
        dur_p95_s=("duration_s", lambda x: x.dropna().quantile(0.95) if x.dropna().size else float("nan")),
    ).reset_index()
    wf_ep.to_csv(WF_EP_OUT, index=False, encoding="utf-8-sig")

    style = top.groupby(["episode_env_styles"]).agg(
        runs=("run_instance_key", "nunique"),
        success_runs=("is_success", "sum"),
        failure_runs=("is_failure", "sum"),
        success_rate=("is_success", "mean"),
        dur_mean_s=("duration_s", "mean"),
        dur_median_s=("duration_s", "median"),
    ).reset_index()
    style.to_csv(STYLE_OUT, index=False, encoding="utf-8-sig")

# ============================================================
# Token loader + repo list loader (unchanged)
# ============================================================
def load_tokens_from_env_file(env_path: Path, token_nums=None) -> List[str]:
    """
    token_nums: iterable of ints, e.g. [4,5,6,7]. If None, loads all tokens.
    """
    wanted = set(token_nums) if token_nums is not None else None
    tokens: List[str] = []

    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")

        m = re.fullmatch(r"GITHUB_TOKEN_(\d+)", k)
        if not m or not v:
            continue

        num = int(m.group(1))
        if wanted is None or num in wanted:
            tokens.append(v)

    if not tokens:
        raise ValueError(f"No tokens found for {sorted(wanted) if wanted else 'ALL'} in {env_path}")
    return tokens


def read_repo_list_from_url_csv(path: Path) -> List[str]:
    df = pd.read_csv(path, encoding="utf-8-sig")
    cols_l = {c.lower(): c for c in df.columns}
    cand_cols = []
    for k in ["html_url", "url", "repo_url", "repository", "repo", "full_name"]:
        if k in cols_l:
            cand_cols.append(cols_l[k])
    if not cand_cols:
        cand_cols = [df.columns[0]]

    out = []
    for col in cand_cols[:1]:
        for v in df[col].astype(str).tolist():
            full = parse_repo_full_name(v)
            if full:
                out.append(full)
    return sorted(set(out))

# ============================================================
# Main
# ============================================================
def main():
    if not URL_LIST_CSV.exists():
        raise FileNotFoundError(f"URL list not found: {URL_LIST_CSV}")
    if not EPISODES_CSV.exists():
        raise FileNotFoundError(f"Episodes file not found: {EPISODES_CSV}")
    if not TOKENS_ENV_PATH.exists():
        raise FileNotFoundError(f"Tokens env not found: {TOKENS_ENV_PATH}")

    cp = load_checkpoint()
    resuming = AUTO_RESUME and CHECKPOINT_PATH.exists() and RAW_OUT.exists()

    if not resuming:
        # fresh run
        if RAW_OUT.exists():
            RAW_OUT.unlink()
        if CHECKPOINT_PATH.exists():
            CHECKPOINT_PATH.unlink()
        cp = load_checkpoint()

    # Optional: de-dupe by existing keys (useful on resume)
    existing_keys = set()
    if resuming:
        try:
            tmp = pd.read_csv(RAW_OUT, usecols=["run_instance_key"], encoding="utf-8-sig")
            existing_keys = set(tmp["run_instance_key"].astype(str).tolist())
            print(f"[resume] loaded {len(existing_keys)} existing run_instance_key values")
        except Exception:
            existing_keys = set()

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, token_nums=[4, 5, 6, 7])

    gh = GitHubClient(tokens)

    repos = read_repo_list_from_url_csv(URL_LIST_CSV)
    repo_map = build_repo_canon_map(repos)
    episodes = load_episodes(EPISODES_CSV, repo_map)

    print(f"[load] repos={len(repos)}")
    print(f"[load] episodes(matched & cutoff<=2025-08-10)={len(episodes)}")

    if len(episodes) == 0:
        print("[ERROR] No episodes matched your URL list (or all are after cutoff).")
        return

    default_branch_cache: Dict[str, str] = {}
    run_cache_by_check_suite: Dict[int, Dict] = {}
    jobs_cache_by_run: Dict[int, List[Dict]] = {}

    for full_name, grp in episodes.groupby("full_name"):
        if full_name not in default_branch_cache:
            default_branch_cache[full_name] = get_repo_default_branch(gh, full_name)
        branch = default_branch_cache[full_name]
        if not branch:
            continue

        for _, ep in grp.iterrows():
            ep_id = ep["episode_id"]
            k = ep_key(full_name, ep_id)
            if cp.get("episode_done", {}).get(k):
                continue  # already finished

            start_dt = ep["episode_start_utc"]
            end_dt   = ep["episode_end_utc"]
            if pd.isna(start_dt) or pd.isna(end_dt) or start_dt >= end_dt:
                continue

            # resume state if exists
            st = cp.get("episode_state", {}).get(k, {})
            cursor = pd.to_datetime(st.get("cursor"), utc=True) if st.get("cursor") else start_dt
            window_days = int(st.get("window_days", INITIAL_WINDOW_DAYS))
            commits_scanned = int(st.get("commits_scanned", 0))
            collected_top = int(st.get("collected_top", 0))

            seen_shas = set()  # we don't persist this; dedupe by run_instance_key instead

            raw_rows: List[Dict] = []

            while (
                cursor < end_dt
                and collected_top < K_INSTRU_RECORDS_PER_EPISODE
                and commits_scanned < MAX_COMMITS_TO_SCAN_PER_EPISODE
            ):
                window_end = min(end_dt, cursor + timedelta(days=window_days))
                since_iso = cursor.isoformat().replace("+00:00", "Z")
                until_iso = window_end.isoformat().replace("+00:00", "Z")

                shas_newest_to_oldest = list_commits_in_range(gh, full_name, branch, since_iso, until_iso)
                if not shas_newest_to_oldest:
                    cursor = window_end
                    window_days = min(MAX_WINDOW_DAYS, max(window_days * 2, window_days + 1))
                    # checkpoint
                    cp.setdefault("episode_state", {})[k] = {
                        "cursor": cursor.isoformat(),
                        "window_days": window_days,
                        "commits_scanned": commits_scanned,
                        "collected_top": collected_top,
                    }
                    save_checkpoint(cp)
                    continue

                shas = list(reversed(shas_newest_to_oldest))  # oldest -> newest

                for sha in shas:
                    if collected_top >= K_INSTRU_RECORDS_PER_EPISODE or commits_scanned >= MAX_COMMITS_TO_SCAN_PER_EPISODE:
                        break
                    if sha in seen_shas:
                        continue
                    seen_shas.add(sha)
                    commits_scanned += 1

                    # ---- Check runs ----
                    for cr in list_check_runs_for_commit(gh, full_name, sha):
                        app = cr.get("app") or {}
                        app_slug = (app.get("slug") or "").strip().lower()

                        if not allow_check_run(app_slug):
                            continue

                        name = (cr.get("name") or "").strip()
                        if not is_instru(name):
                            continue

                        started_at = cr.get("started_at") or ""
                        completed_at = cr.get("completed_at") or ""
                        dur = seconds_between(started_at, completed_at)
                        concl = cr.get("conclusion") or cr.get("status") or ""

                        run_key = f"checkrun:{cr.get('id')}"
                        if run_key in existing_keys:
                            continue
                        existing_keys.add(run_key)

                        is_gha = (app_slug == "github-actions")

                        raw_rows.append({
                            "full_name": full_name,
                            "default_branch": branch,
                            "episode_id": ep_id,
                            "episode_env_styles": ep["episode_env_styles"],
                            "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                            "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                            "commit_sha": sha,

                            "record_type": "check_run",
                            "level": "check_run",
                            "provider": app_slug or "unknown_app",
                            "provider_kind": "github_check_run",
                            "is_github_actions": "yes" if is_gha else "no",

                            "job_name": name,
                            "job_conclusion": concl,

                            "started_at": started_at,
                            "completed_at": completed_at,
                            "duration_seconds": dur if dur is not None else "",

                            "html_url": cr.get("html_url") or "",
                            "details_url": cr.get("details_url") or "",
                            "collected_at_utc": now_utc_iso(),
                            "run_instance_key": run_key,
                        })
                        collected_top += 1

                        # ---- NEW: expand GitHub Actions run -> jobs -> steps (if possible) ----
                        if is_gha:
                            check_suite = cr.get("check_suite") or {}
                            cs_id = check_suite.get("id")
                            if isinstance(cs_id, int):
                                if cs_id not in run_cache_by_check_suite:
                                    run_obj = get_workflow_run_by_check_suite_id(gh, full_name, cs_id)
                                    if isinstance(run_obj, dict):
                                        run_cache_by_check_suite[cs_id] = run_obj
                                run_obj = run_cache_by_check_suite.get(cs_id)

                                if isinstance(run_obj, dict):
                                    run_id = run_obj.get("id")
                                    attempt = run_obj.get("run_attempt") or 1

                                    # write run row
                                    rkey = f"gharun:{run_id}:{attempt}"
                                    if rkey not in existing_keys and isinstance(run_id, int):
                                        existing_keys.add(rkey)
                                        raw_rows.append({
                                            "full_name": full_name,
                                            "default_branch": branch,
                                            "episode_id": ep_id,
                                            "episode_env_styles": ep["episode_env_styles"],
                                            "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                                            "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                                            "commit_sha": sha,

                                            "record_type": "gha_run",
                                            "level": "run",
                                            "provider": "github-actions",
                                            "provider_kind": "github_actions",
                                            "is_github_actions": "yes",

                                            "workflow_run_id": run_id,
                                            "workflow_run_attempt": attempt,
                                            "workflow_name": run_obj.get("name") or "",
                                            "workflow_event": run_obj.get("event") or "",
                                            "head_branch": run_obj.get("head_branch") or "",
                                            "head_sha": run_obj.get("head_sha") or "",
                                            "run_created_at": run_obj.get("created_at") or "",
                                            "run_updated_at": run_obj.get("updated_at") or "",

                                            "job_status": run_obj.get("status") or "",
                                            "job_conclusion": run_obj.get("conclusion") or "",

                                            "html_url": run_obj.get("html_url") or "",
                                            "details_url": "",
                                            "collected_at_utc": now_utc_iso(),
                                            "run_instance_key": rkey,
                                        })

                                    # fetch jobs (cached)
                                    if isinstance(run_id, int):
                                        if run_id not in jobs_cache_by_run:
                                            jobs_cache_by_run[run_id] = list_jobs_for_run(gh, full_name, run_id)
                                        jobs = jobs_cache_by_run.get(run_id, [])

                                        for j in jobs:
                                            j_id = j.get("id")
                                            j_name = j.get("name") or ""
                                            # keep only instrumentation-ish jobs OR jobs containing instrumentation-ish steps
                                            steps = j.get("steps") or []
                                            job_match = is_instru(j_name) or any(is_instru((s.get("name") or "")) for s in steps)

                                            if not job_match:
                                                continue

                                            jkey = f"ghajob:{j_id}"
                                            if jkey not in existing_keys and isinstance(j_id, int):
                                                existing_keys.add(jkey)
                                                raw_rows.append({
                                                    "full_name": full_name,
                                                    "default_branch": branch,
                                                    "episode_id": ep_id,
                                                    "episode_env_styles": ep["episode_env_styles"],
                                                    "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                                                    "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                                                    "commit_sha": sha,

                                                    "record_type": "gha_job",
                                                    "level": "job",
                                                    "provider": "github-actions",
                                                    "provider_kind": "github_actions",
                                                    "is_github_actions": "yes",

                                                    "workflow_run_id": run_id,
                                                    "workflow_run_attempt": attempt,

                                                    "job_id": j_id,
                                                    "job_name": j_name,
                                                    "job_status": j.get("status") or "",
                                                    "job_conclusion": j.get("conclusion") or "",
                                                    "runner_name": j.get("runner_name") or "",
                                                    "runner_labels": ",".join(j.get("labels") or []) if isinstance(j.get("labels"), list) else "",

                                                    "started_at": j.get("started_at") or "",
                                                    "completed_at": j.get("completed_at") or "",
                                                    "duration_seconds": seconds_between(j.get("started_at"), j.get("completed_at")) or "",

                                                    "html_url": j.get("html_url") or "",
                                                    "details_url": "",
                                                    "collected_at_utc": now_utc_iso(),
                                                    "run_instance_key": jkey,
                                                })

                                            # steps
                                            for s in steps:
                                                s_name = s.get("name") or ""
                                                if not is_instru(s_name):
                                                    continue

                                                s_num = s.get("number") or ""
                                                skey = f"ghastep:{j_id}:{s_num}:{s_name}"
                                                if skey in existing_keys:
                                                    continue
                                                existing_keys.add(skey)

                                                raw_rows.append({
                                                    "full_name": full_name,
                                                    "default_branch": branch,
                                                    "episode_id": ep_id,
                                                    "episode_env_styles": ep["episode_env_styles"],
                                                    "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                                                    "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                                                    "commit_sha": sha,

                                                    "record_type": "gha_step",
                                                    "level": "step",
                                                    "provider": "github-actions",
                                                    "provider_kind": "github_actions",
                                                    "is_github_actions": "yes",

                                                    "workflow_run_id": run_id,
                                                    "workflow_run_attempt": attempt,

                                                    "job_id": j_id,
                                                    "job_name": j_name,

                                                    "step_number": s_num,
                                                    "step_name": s_name,
                                                    "step_status": s.get("status") or "",
                                                    "step_conclusion": s.get("conclusion") or "",

                                                    "started_at": s.get("started_at") or "",
                                                    "completed_at": s.get("completed_at") or "",
                                                    "duration_seconds": seconds_between(s.get("started_at"), s.get("completed_at")) or "",

                                                    "html_url": "",
                                                    "details_url": "",
                                                    "collected_at_utc": now_utc_iso(),
                                                    "run_instance_key": skey,
                                                })

                        if collected_top >= K_INSTRU_RECORDS_PER_EPISODE:
                            break

                    if collected_top >= K_INSTRU_RECORDS_PER_EPISODE:
                        break

                    # ---- Commit statuses (external CI often shows up here) ----
                    if allow_commit_status():
                        st0 = get_combined_status_for_commit(gh, full_name, sha)
                        statuses = st0.get("statuses", []) if isinstance(st0, dict) else []
                        for s in statuses:
                            context = (s.get("context") or "").strip()
                            if not is_instru(context):
                                continue

                            state = (s.get("state") or "").strip().lower()
                            created_at = s.get("created_at") or ""
                            updated_at = s.get("updated_at") or ""
                            dur = seconds_between(created_at, updated_at)

                            prov = (context.split("/")[0] if "/" in context else context.split(":")[0]).strip().lower()

                            skey = f"status:{sha}:{context}:{created_at}"
                            if skey in existing_keys:
                                continue
                            existing_keys.add(skey)

                            raw_rows.append({
                                "full_name": full_name,
                                "default_branch": branch,
                                "episode_id": ep_id,
                                "episode_env_styles": ep["episode_env_styles"],
                                "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                                "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                                "commit_sha": sha,

                                "record_type": "commit_status",
                                "level": "commit_status",
                                "provider": prov or "not_github_actions",
                                "provider_kind": "github_commit_status",
                                "is_github_actions": "no",

                                "job_name": context,
                                "job_conclusion": state,

                                "started_at": created_at,
                                "completed_at": updated_at,
                                "duration_seconds": dur if dur is not None else "",

                                "html_url": s.get("target_url") or "",
                                "details_url": "",
                                "collected_at_utc": now_utc_iso(),
                                "run_instance_key": skey,
                            })
                            collected_top += 1
                            if collected_top >= K_INSTRU_RECORDS_PER_EPISODE:
                                break

                # flush window chunk (so you see progress even mid-episode)
                flush_rows(raw_rows, RAW_OUT)
                raw_rows.clear()

                cursor = window_end
                window_days = min(MAX_WINDOW_DAYS, max(window_days * 2, window_days + 1))

                # checkpoint
                cp.setdefault("episode_state", {})[k] = {
                    "cursor": cursor.isoformat(),
                    "window_days": window_days,
                    "commits_scanned": commits_scanned,
                    "collected_top": collected_top,
                }
                save_checkpoint(cp)

            print(f"[episode] {full_name} ep={ep_id} style={ep['episode_env_styles']} "
                  f"collected_top={collected_top} commits_scanned={commits_scanned}")

            # mark done + recompute aggregates so files appear during long runs
            cp.setdefault("episode_done", {})[k] = True
            cp.get("episode_state", {}).pop(k, None)
            save_checkpoint(cp)

            recompute_aggregates()
            print(f"[save] aggregates updated: {WF_EP_OUT.name}, {STYLE_OUT.name}")

    # final aggregates
    recompute_aggregates()
    print("Done.")

if __name__ == "__main__":
    main()


[load] repos=481
[load] episodes(matched & cutoff<=2025-08-10)=521
[episode] 4eRTuk/audioview ep=1 style=Emu_Custom collected_top=0 commits_scanned=39
[save] aggregates updated: routeA_workflow_episode_metrics.csv, routeA_style_overall_metrics.csv
[episode] AAkira/ExpandableLayout ep=1 style=Emu_Custom collected_top=0 commits_scanned=97
[save] aggregates updated: routeA_workflow_episode_metrics.csv, routeA_style_overall_metrics.csv
[episode] AChep/AcDisplay ep=1 style=Emu_Custom collected_top=0 commits_scanned=898
[save] aggregates updated: routeA_workflow_episode_metrics.csv, routeA_style_overall_metrics.csv
[episode] ActivityWatch/aw-android ep=1 style=Emu_Custom collected_top=0 commits_scanned=75
[save] aggregates updated: routeA_workflow_episode_metrics.csv, routeA_style_overall_metrics.csv
[episode] AdamMc331/AndroidStudyGuide ep=1 style=Emu_Community collected_top=0 commits_scanned=18
[save] aggregates updated: routeA_workflow_episode_metrics.csv, routeA_style_overall_metrics.csv

In [ ]:
################### V2.0 #################
# small change from above: includes the records of not matched so we can distinguish when not availabel from is not matched

#the count of unmatched check-runs/statuses too, so you can distinguish:

#“no CI signals exist on GitHub” vs
#E“signals exist but my regex filtered them out.”

In [ ]:
import csv
import json
import os
import re
import time
import random
from dataclasses import dataclass
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Union, Tuple

import requests
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
ROOT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_B")

TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

URL_LIST_CSV = ROOT / "URL_List_Instru.csv"
EPISODES_CSV  = ROOT / "List_Change_Episodes.csv"

OUT_DIR = ROOT / "RouteA_EpisodeRunMetrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_OUT = OUT_DIR / "routeA_runs_labeled.csv"
WF_EP_OUT = OUT_DIR / "routeA_workflow_episode_metrics.csv"
STYLE_OUT = OUT_DIR / "routeA_style_overall_metrics.csv"

# NEW: “net-of-skipped” outputs
WF_EP_OUT_EXEC = OUT_DIR / "routeA_workflow_episode_metrics_executed.csv"
STYLE_OUT_EXEC = OUT_DIR / "routeA_style_overall_metrics_executed.csv"

# NEW: per-episode coverage diagnostics (helps explain missing episodes)
EP_COV_OUT = OUT_DIR / "routeA_episode_coverage.csv"

CHECKPOINT_PATH = OUT_DIR / "routeA_checkpoint.json"

# Hard cutoff (inclusive end-of-day UTC)
CUTOFF_UTC = datetime(2025, 8, 10, 23, 59, 59, tzinfo=timezone.utc)

# Resume automatically if checkpoint exists
AUTO_RESUME = True

# Stop condition: collect K top-level instrumentation CI *matches* per episode
# NOTE: Steps can add extra rows; K only controls “top-level” matches (check_run + commit_status).
K_INSTRU_RECORDS_PER_EPISODE = 100

INITIAL_WINDOW_DAYS = 30
MAX_WINDOW_DAYS = 3650
MAX_COMMITS_TO_SCAN_PER_EPISODE = 5000

PROVIDER_FOCUS = "both"  # gha_only | non_gha_only | both

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

# NEW: tighten “episode window” claim by filtering run timestamps (recommended)
# If True: only count a top-level record toward K when its timestamps are within the episode.
STRICT_EPISODE_TIME_FILTER = True
# If timestamps are missing, should we allow the record to count toward K?
ALLOW_MISSING_TIMESTAMPS_WHEN_STRICT = False

# ============================================================
# Instrumentation-ish matching
# ============================================================
INSTRU_NAME_RE = re.compile(
    r"(androidtest|connectedandroidtest|connected.*android.*test|connectedcheck|devicecheck|"
    r"manageddevice|gmd|instrument(ed|ation)?|am\s+instrument|espresso|uiautomator|ui\s*test|"
    r"firebase\s+test|test\s*lab|device\s*farm|browserstack|sauce(labs)?|kobiton|appcenter|"
    r"flank|maestro|marathon|spoon|baselineprofile|benchmark)",
    re.IGNORECASE,
)

def is_instru(s: str) -> bool:
    return bool(INSTRU_NAME_RE.search(s or ""))

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def to_dt_utc(x: Optional[str]) -> Optional[datetime]:
    if not x:
        return None
    try:
        d = pd.to_datetime(x, utc=True, errors="coerce")
        if pd.isna(d):
            return None
        return d.to_pydatetime()
    except Exception:
        return None

def seconds_between(a: Optional[str], b: Optional[str]) -> Optional[int]:
    try:
        da = pd.to_datetime(a, utc=True)
        db = pd.to_datetime(b, utc=True)
        sec = int((db - da).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None

def in_episode_window(ts: Optional[str], start_dt: datetime, end_dt: datetime) -> Optional[bool]:
    d = to_dt_utc(ts)
    if d is None:
        return None
    return (start_dt <= d <= end_dt)

def time_filter_pass(start_ts: Optional[str], end_ts: Optional[str], start_dt: datetime, end_dt: datetime) -> bool:
    """
    Strict test: if both timestamps exist, require both within [start_dt, end_dt].
    If one/both are missing: return ALLOW_MISSING_TIMESTAMPS_WHEN_STRICT.
    """
    a_ok = in_episode_window(start_ts, start_dt, end_dt)
    b_ok = in_episode_window(end_ts, start_dt, end_dt)
    if a_ok is None or b_ok is None:
        return ALLOW_MISSING_TIMESTAMPS_WHEN_STRICT
    return bool(a_ok and b_ok)

# ============================================================
# Output schema (fixed header!)
# ============================================================
FIELDNAMES = [
    "full_name","default_branch",
    "episode_id","episode_env_styles","episode_start_utc","episode_end_utc",
    "commit_sha",

    # general record identity
    "record_type","level",
    "provider","provider_kind","is_github_actions",

    # workflow run
    "workflow_run_id","workflow_run_attempt","workflow_name","workflow_event",
    "head_branch","head_sha","run_created_at","run_updated_at",

    # job
    "job_id","job_name","job_status","job_conclusion","runner_name","runner_labels",

    # step
    "step_number","step_name","step_status","step_conclusion",

    # timings (common fields)
    "started_at","completed_at","duration_seconds",

    # urls + bookkeeping
    "html_url","details_url",
    "collected_at_utc","run_instance_key",

    # NEW: window-claim diagnostics
    "started_in_episode","completed_in_episode","strict_time_filter_pass",
]

def _row_with_defaults(d: Dict) -> Dict:
    out = {k: "" for k in FIELDNAMES}
    for k, v in d.items():
        if k in out:
            out[k] = v
    return out

def flush_rows(rows: List[Dict], out_path: Path):
    if not rows:
        return
    write_header = not out_path.exists()
    with out_path.open("a", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=FIELDNAMES, extrasaction="ignore")
        if write_header:
            w.writeheader()
        for r in rows:
            w.writerow(_row_with_defaults(r))

# ============================================================
# Resume / checkpoint
# ============================================================
def load_checkpoint() -> Dict:
    if not CHECKPOINT_PATH.exists():
        return {"version": 2, "cutoff_utc": CUTOFF_UTC.isoformat(), "episode_state": {}, "episode_done": {}}
    try:
        return json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
    except Exception:
        return {"version": 2, "cutoff_utc": CUTOFF_UTC.isoformat(), "episode_state": {}, "episode_done": {}}

def save_checkpoint(cp: Dict) -> None:
    cp["last_updated_utc"] = now_utc_iso()
    CHECKPOINT_PATH.write_text(json.dumps(cp, indent=2, sort_keys=True), encoding="utf-8")

def ep_key(full_name: str, episode_id: Union[str,int]) -> str:
    return f"{full_name}||{episode_id}"

# ============================================================
# Repo parsing helpers
# ============================================================
def parse_repo_full_name(repo_url_or_fullname: str) -> Optional[str]:
    s = (repo_url_or_fullname or "").strip()
    if not s:
        return None
    if re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", s):
        return s
    m = re.search(r"github\.com[:/]+([A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+)", s, re.IGNORECASE)
    if m:
        full = m.group(1)
        return full[:-4] if full.endswith(".git") else full
    return None

def allow_check_run(app_slug: str) -> bool:
    slug = (app_slug or "").strip().lower()
    if PROVIDER_FOCUS == "both":
        return True
    if PROVIDER_FOCUS == "gha_only":
        return slug == "github-actions"
    if PROVIDER_FOCUS == "non_gha_only":
        return slug != "github-actions"
    return True

def allow_commit_status() -> bool:
    return PROVIDER_FOCUS != "gha_only"

# ============================================================
# GitHub client
# ============================================================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "episode-ci-metrics-k/2.1",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        time.sleep(max(1, soonest - now + 2))

    def _backoff(self, attempt: int) -> None:
        time.sleep(min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random())

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code
            try:
                st.remaining = int(resp.headers.get("X-RateLimit-Remaining", "0"))
            except Exception:
                pass
            try:
                st.reset_epoch = int(resp.headers.get("X-RateLimit-Reset", "0"))
            except Exception:
                pass

            if resp.status_code == 404:
                return None

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and ("rate limit" in text_l or "secondary rate limit" in text_l):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate_list(self, url: str, params: Optional[Dict] = None) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if not data or not isinstance(data, list):
                return
            if not data:
                return
            for it in data:
                yield it
            if len(data) < 100:
                return
            page += 1

# ============================================================
# GitHub endpoints
# ============================================================
def get_repo_default_branch(gh: GitHubClient, full_name: str) -> str:
    url = f"https://api.github.com/repos/{full_name}"
    data = gh.request_json("GET", url)
    if not data or not isinstance(data, dict):
        return ""
    return (data.get("default_branch") or "").strip()

def list_commits_in_range(gh: GitHubClient, full_name: str, branch: str, since_iso: str, until_iso: str) -> List[str]:
    url = f"https://api.github.com/repos/{full_name}/commits"
    params = {"sha": branch, "since": since_iso, "until": until_iso}
    shas: List[str] = []
    for c in gh.paginate_list(url, params=params):
        sha = c.get("sha")
        if isinstance(sha, str) and len(sha) >= 7:
            shas.append(sha)
    return shas  # newest -> oldest

def list_check_runs_for_commit(gh: GitHubClient, full_name: str, sha: str) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/commits/{sha}/check-runs"
    data = gh.request_json("GET", url)
    if not data or not isinstance(data, dict):
        return []
    return data.get("check_runs", []) or []

def get_combined_status_for_commit(gh: GitHubClient, full_name: str, sha: str) -> Dict:
    url = f"https://api.github.com/repos/{full_name}/commits/{sha}/status"
    data = gh.request_json("GET", url)
    return data if isinstance(data, dict) else {}

# ============================================================
# GitHub Actions run -> jobs -> steps (tied to check_suite_id)
# ============================================================
def get_workflow_run_by_check_suite_id(gh: GitHubClient, full_name: str, check_suite_id: int) -> Optional[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs"
    params = {"check_suite_id": check_suite_id, "per_page": 1, "page": 1}
    data = gh.request_json("GET", url, params=params)
    if not data or not isinstance(data, dict):
        return None
    runs = data.get("workflow_runs") or []
    return runs[0] if runs else None

def list_jobs_for_run(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    out: List[Dict] = []
    page = 1
    while page <= MAX_PAGES_PER_LIST:
        params = {"per_page": 100, "page": page}
        data = gh.request_json("GET", url, params=params)
        if not data or not isinstance(data, dict):
            break
        jobs = data.get("jobs") or []
        if not jobs:
            break
        out.extend(jobs)
        if len(jobs) < 100:
            break
        page += 1
    return out

# ============================================================
# Episodes loader (with cutoff)
# ============================================================
def build_repo_canon_map(repos: List[str]) -> Dict[str, str]:
    return {r.lower(): r for r in repos}

def normalize_episode_repo(row: pd.Series, repo_map: Dict[str, str]) -> Optional[str]:
    candidates: List[str] = []
    for key in ["repo_name", "full_name", "html_url", "url", "repo", "repository", "repo_url"]:
        if key in row and pd.notna(row[key]):
            s = str(row[key]).strip()
            if s:
                candidates.append(s)

    for raw in candidates:
        p = parse_repo_full_name(raw)
        if p and p.lower() in repo_map:
            return repo_map[p.lower()]

        if "__" in raw and raw.count("__") == 1:
            p2 = raw.replace("__", "/")
            if p2.lower() in repo_map:
                return repo_map[p2.lower()]

        if "/" not in raw and "." in raw:
            p3 = raw.replace(".", "/", 1)
            if p3.lower() in repo_map:
                return repo_map[p3.lower()]

    return None

def load_episodes(path: Path, repo_map: Dict[str, str]) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")

    if "episode_start_utc" not in df.columns or "episode_end_utc" not in df.columns:
        raise ValueError(f"Missing episode_start_utc/episode_end_utc in {path.name}. Columns={list(df.columns)}")

    out = df.copy()
    out["episode_start_utc"] = pd.to_datetime(out["episode_start_utc"], utc=True, errors="coerce")
    out["episode_end_utc"]   = pd.to_datetime(out["episode_end_utc"], utc=True, errors="coerce")

    style_col = "env_styles" if "env_styles" in out.columns else None
    out["episode_env_styles"] = out[style_col].astype(str) if style_col else ""

    if "episode_id" in out.columns:
        out["episode_id"] = out["episode_id"]
    elif "episode_index" in out.columns:
        out["episode_id"] = out["episode_index"]
    else:
        out["episode_id"] = range(len(out))

    out["full_name"] = out.apply(lambda r: normalize_episode_repo(r, repo_map), axis=1)
    out = out.dropna(subset=["full_name", "episode_start_utc", "episode_end_utc"])

    # apply cutoff: skip episodes fully after cutoff, and cap end at cutoff
    out = out[out["episode_start_utc"] <= CUTOFF_UTC].copy()
    out["episode_end_utc"] = out["episode_end_utc"].apply(lambda d: min(d, CUTOFF_UTC) if pd.notna(d) else d)
    out = out[out["episode_start_utc"] < out["episode_end_utc"]].copy()

    return out[["full_name", "episode_id", "episode_start_utc", "episode_end_utc", "episode_env_styles"]]

# ============================================================
# Aggregations
# ============================================================
SUCCESS_SET = {"success", "neutral"}  # you can tweak
FAIL_SET = {"failure", "error", "cancelled", "timed_out", "action_required"}
SKIP_SET = {"skipped"}

def recompute_aggregates():
    if not RAW_OUT.exists():
        return
    df = pd.read_csv(RAW_OUT, encoding="utf-8-sig")
    if len(df) == 0:
        return

    # Only aggregate “top-level CI” rows (avoid counting steps as runs)
    top = df[df["record_type"].isin(["check_run","commit_status"])].copy()
    if len(top) == 0:
        return

    concl = top["job_conclusion"].astype(str).str.lower().fillna("")
    top["is_success_incl_skipped"] = concl.isin(SUCCESS_SET.union(SKIP_SET))
    top["is_failure"] = concl.isin(FAIL_SET)
    top["is_skipped"] = concl.isin(SKIP_SET)

    top["duration_s"] = pd.to_numeric(top["duration_seconds"], errors="coerce")

    gcols = ["full_name", "episode_id", "episode_env_styles", "provider_kind", "provider", "job_name"]

    # (A) Original-style metrics (skipped treated as success)
    wf_ep = top.groupby(gcols).agg(
        runs=("run_instance_key", "nunique"),
        success_runs=("is_success_incl_skipped", "sum"),
        failure_runs=("is_failure", "sum"),
        skipped_runs=("is_skipped", "sum"),
        success_rate=("is_success_incl_skipped", "mean"),
        skip_rate=("is_skipped", "mean"),
        dur_mean_s=("duration_s", "mean"),
        dur_median_s=("duration_s", "median"),
        dur_p95_s=("duration_s", lambda x: x.dropna().quantile(0.95) if x.dropna().size else float("nan")),
    ).reset_index()
    wf_ep.to_csv(WF_EP_OUT, index=False, encoding="utf-8-sig")

    style = top.groupby(["episode_env_styles"]).agg(
        runs=("run_instance_key", "nunique"),
        success_runs=("is_success_incl_skipped", "sum"),
        failure_runs=("is_failure", "sum"),
        skipped_runs=("is_skipped", "sum"),
        success_rate=("is_success_incl_skipped", "mean"),
        skip_rate=("is_skipped", "mean"),
        dur_mean_s=("duration_s", "mean"),
        dur_median_s=("duration_s", "median"),
    ).reset_index()
    style.to_csv(STYLE_OUT, index=False, encoding="utf-8-sig")

    # (B) “Executed only” metrics (net of skipped)
    executed = top[~top["is_skipped"]].copy()
    if len(executed) > 0:
        concl2 = executed["job_conclusion"].astype(str).str.lower().fillna("")
        executed["is_success_exec"] = concl2.isin(SUCCESS_SET)
        executed["is_failure_exec"] = concl2.isin(FAIL_SET)
        executed["duration_s"] = pd.to_numeric(executed["duration_seconds"], errors="coerce")

        wf_ep_exec = executed.groupby(gcols).agg(
            executed_runs=("run_instance_key", "nunique"),
            success_runs=("is_success_exec", "sum"),
            failure_runs=("is_failure_exec", "sum"),
            success_rate=("is_success_exec", "mean"),
            dur_mean_s=("duration_s", "mean"),
            dur_median_s=("duration_s", "median"),
            dur_p95_s=("duration_s", lambda x: x.dropna().quantile(0.95) if x.dropna().size else float("nan")),
        ).reset_index()
        wf_ep_exec.to_csv(WF_EP_OUT_EXEC, index=False, encoding="utf-8-sig")

        style_exec = executed.groupby(["episode_env_styles"]).agg(
            executed_runs=("run_instance_key", "nunique"),
            success_runs=("is_success_exec", "sum"),
            failure_runs=("is_failure_exec", "sum"),
            success_rate=("is_success_exec", "mean"),
            dur_mean_s=("duration_s", "mean"),
            dur_median_s=("duration_s", "median"),
        ).reset_index()
        style_exec.to_csv(STYLE_OUT_EXEC, index=False, encoding="utf-8-sig")
    else:
        # still write empty headers for consistency
        pd.DataFrame(columns=gcols + ["executed_runs","success_runs","failure_runs","success_rate","dur_mean_s","dur_median_s","dur_p95_s"]) \
          .to_csv(WF_EP_OUT_EXEC, index=False, encoding="utf-8-sig")
        pd.DataFrame(columns=["episode_env_styles","executed_runs","success_runs","failure_runs","success_rate","dur_mean_s","dur_median_s"]) \
          .to_csv(STYLE_OUT_EXEC, index=False, encoding="utf-8-sig")

# ============================================================
# Token loader + repo list loader
# ============================================================
def load_tokens_from_env_file(env_path: Path, token_nums=None) -> List[str]:
    wanted = set(token_nums) if token_nums is not None else None
    tokens: List[str] = []

    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")

        m = re.fullmatch(r"GITHUB_TOKEN_(\d+)", k)
        if not m or not v:
            continue

        num = int(m.group(1))
        if wanted is None or num in wanted:
            tokens.append(v)

    if not tokens:
        raise ValueError(f"No tokens found for {sorted(wanted) if wanted else 'ALL'} in {env_path}")
    return tokens

def read_repo_list_from_url_csv(path: Path) -> List[str]:
    df = pd.read_csv(path, encoding="utf-8-sig")
    cols_l = {c.lower(): c for c in df.columns}
    cand_cols = []
    for k in ["html_url", "url", "repo_url", "repository", "repo", "full_name"]:
        if k in cols_l:
            cand_cols.append(cols_l[k])
    if not cand_cols:
        cand_cols = [df.columns[0]]

    out = []
    for col in cand_cols[:1]:
        for v in df[col].astype(str).tolist():
            full = parse_repo_full_name(v)
            if full:
                out.append(full)
    return sorted(set(out))

# ============================================================
# Episode coverage diagnostics
# ============================================================
EP_COV_FIELDS = [
    "full_name","episode_id","episode_env_styles","episode_start_utc","episode_end_utc",
    "commits_scanned","windows_advanced",
    "checkruns_seen_total","checkruns_instru_name","checkruns_timefilter_pass",
    "statuses_seen_total","statuses_instru_context","statuses_timefilter_pass",
    "collected_top_final",
    "collected_check_run_top","collected_commit_status_top",
]

def append_episode_coverage(row: Dict):
    write_header = not EP_COV_OUT.exists()
    with EP_COV_OUT.open("a", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=EP_COV_FIELDS, extrasaction="ignore")
        if write_header:
            w.writeheader()
        w.writerow({k: row.get(k, "") for k in EP_COV_FIELDS})

# ============================================================
# Main
# ============================================================
def main():
    if not URL_LIST_CSV.exists():
        raise FileNotFoundError(f"URL list not found: {URL_LIST_CSV}")
    if not EPISODES_CSV.exists():
        raise FileNotFoundError(f"Episodes file not found: {EPISODES_CSV}")
    if not TOKENS_ENV_PATH.exists():
        raise FileNotFoundError(f"Tokens env not found: {TOKENS_ENV_PATH}")

    cp = load_checkpoint()
    resuming = AUTO_RESUME and CHECKPOINT_PATH.exists() and RAW_OUT.exists()

    if not resuming:
        # fresh run
        if RAW_OUT.exists():
            RAW_OUT.unlink()
        if CHECKPOINT_PATH.exists():
            CHECKPOINT_PATH.unlink()
        if EP_COV_OUT.exists():
            EP_COV_OUT.unlink()
        cp = load_checkpoint()

    # Optional: de-dupe by existing keys (useful on resume)
    existing_keys = set()
    if resuming:
        try:
            tmp = pd.read_csv(RAW_OUT, usecols=["run_instance_key"], encoding="utf-8-sig")
            existing_keys = set(tmp["run_instance_key"].astype(str).tolist())
            print(f"[resume] loaded {len(existing_keys)} existing run_instance_key values")
        except Exception:
            existing_keys = set()

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, token_nums=[4, 5, 6, 7])
    gh = GitHubClient(tokens)

    repos = read_repo_list_from_url_csv(URL_LIST_CSV)
    repo_map = build_repo_canon_map(repos)
    episodes = load_episodes(EPISODES_CSV, repo_map)

    print(f"[load] repos={len(repos)}")
    print(f"[load] episodes(matched & cutoff<=2025-08-10)={len(episodes)}")

    if len(episodes) == 0:
        print("[ERROR] No episodes matched your URL list (or all are after cutoff).")
        return

    default_branch_cache: Dict[str, str] = {}
    run_cache_by_check_suite: Dict[int, Dict] = {}
    jobs_cache_by_run: Dict[int, List[Dict]] = {}

    for full_name, grp in episodes.groupby("full_name"):
        if full_name not in default_branch_cache:
            default_branch_cache[full_name] = get_repo_default_branch(gh, full_name)
        branch = default_branch_cache[full_name]
        if not branch:
            continue

        for _, ep in grp.iterrows():
            ep_id = ep["episode_id"]
            k = ep_key(full_name, ep_id)
            if cp.get("episode_done", {}).get(k):
                continue  # already finished

            start_dt = ep["episode_start_utc"]
            end_dt   = ep["episode_end_utc"]
            if pd.isna(start_dt) or pd.isna(end_dt) or start_dt >= end_dt:
                continue

            # resume state if exists
            st = cp.get("episode_state", {}).get(k, {})
            cursor = pd.to_datetime(st.get("cursor"), utc=True) if st.get("cursor") else start_dt
            window_days = int(st.get("window_days", INITIAL_WINDOW_DAYS))
            commits_scanned = int(st.get("commits_scanned", 0))
            collected_top = int(st.get("collected_top", 0))
            windows_advanced = int(st.get("windows_advanced", 0))

            seen_shas = set()

            # diagnostics counters for this episode
            checkruns_seen_total = int(st.get("checkruns_seen_total", 0))
            checkruns_instru_name = int(st.get("checkruns_instru_name", 0))
            checkruns_timefilter_pass = int(st.get("checkruns_timefilter_pass", 0))

            statuses_seen_total = int(st.get("statuses_seen_total", 0))
            statuses_instru_context = int(st.get("statuses_instru_context", 0))
            statuses_timefilter_pass = int(st.get("statuses_timefilter_pass", 0))

            collected_check_run_top = int(st.get("collected_check_run_top", 0))
            collected_commit_status_top = int(st.get("collected_commit_status_top", 0))

            raw_rows: List[Dict] = []

            while (
                cursor < end_dt
                and collected_top < K_INSTRU_RECORDS_PER_EPISODE
                and commits_scanned < MAX_COMMITS_TO_SCAN_PER_EPISODE
            ):
                window_end = min(end_dt, cursor + timedelta(days=window_days))
                since_iso = cursor.isoformat().replace("+00:00", "Z")
                until_iso = window_end.isoformat().replace("+00:00", "Z")

                shas_newest_to_oldest = list_commits_in_range(gh, full_name, branch, since_iso, until_iso)
                if not shas_newest_to_oldest:
                    cursor = window_end
                    window_days = min(MAX_WINDOW_DAYS, max(window_days * 2, window_days + 1))
                    windows_advanced += 1
                    # checkpoint
                    cp.setdefault("episode_state", {})[k] = {
                        "cursor": cursor.isoformat(),
                        "window_days": window_days,
                        "commits_scanned": commits_scanned,
                        "collected_top": collected_top,
                        "windows_advanced": windows_advanced,
                        "checkruns_seen_total": checkruns_seen_total,
                        "checkruns_instru_name": checkruns_instru_name,
                        "checkruns_timefilter_pass": checkruns_timefilter_pass,
                        "statuses_seen_total": statuses_seen_total,
                        "statuses_instru_context": statuses_instru_context,
                        "statuses_timefilter_pass": statuses_timefilter_pass,
                        "collected_check_run_top": collected_check_run_top,
                        "collected_commit_status_top": collected_commit_status_top,
                    }
                    save_checkpoint(cp)
                    continue

                shas = list(reversed(shas_newest_to_oldest))  # oldest -> newest

                for sha in shas:
                    if collected_top >= K_INSTRU_RECORDS_PER_EPISODE or commits_scanned >= MAX_COMMITS_TO_SCAN_PER_EPISODE:
                        break
                    if sha in seen_shas:
                        continue
                    seen_shas.add(sha)
                    commits_scanned += 1

                    # ---- Check runs ----
                    for cr in list_check_runs_for_commit(gh, full_name, sha):
                        app = cr.get("app") or {}
                        app_slug = (app.get("slug") or "").strip().lower()

                        if not allow_check_run(app_slug):
                            continue

                        checkruns_seen_total += 1

                        name = (cr.get("name") or "").strip()
                        if not is_instru(name):
                            continue

                        checkruns_instru_name += 1

                        started_at = cr.get("started_at") or ""
                        completed_at = cr.get("completed_at") or ""
                        dur = seconds_between(started_at, completed_at)
                        concl = cr.get("conclusion") or cr.get("status") or ""

                        strict_pass = True
                        if STRICT_EPISODE_TIME_FILTER:
                            strict_pass = time_filter_pass(started_at, completed_at, start_dt.to_pydatetime(), end_dt.to_pydatetime())

                        if strict_pass:
                            checkruns_timefilter_pass += 1

                        # If strict filter is enabled and fails, do not count toward K and do not write the row
                        if STRICT_EPISODE_TIME_FILTER and not strict_pass:
                            continue

                        run_key = f"checkrun:{cr.get('id')}"
                        if run_key in existing_keys:
                            continue
                        existing_keys.add(run_key)

                        is_gha = (app_slug == "github-actions")

                        raw_rows.append({
                            "full_name": full_name,
                            "default_branch": branch,
                            "episode_id": ep_id,
                            "episode_env_styles": ep["episode_env_styles"],
                            "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                            "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                            "commit_sha": sha,

                            "record_type": "check_run",
                            "level": "check_run",
                            "provider": app_slug or "unknown_app",
                            "provider_kind": "github_check_run",
                            "is_github_actions": "yes" if is_gha else "no",

                            "job_name": name,
                            "job_conclusion": concl,

                            "started_at": started_at,
                            "completed_at": completed_at,
                            "duration_seconds": dur if dur is not None else "",

                            "html_url": cr.get("html_url") or "",
                            "details_url": cr.get("details_url") or "",
                            "collected_at_utc": now_utc_iso(),
                            "run_instance_key": run_key,

                            "started_in_episode": in_episode_window(started_at, start_dt.to_pydatetime(), end_dt.to_pydatetime()),
                            "completed_in_episode": in_episode_window(completed_at, start_dt.to_pydatetime(), end_dt.to_pydatetime()),
                            "strict_time_filter_pass": "yes" if strict_pass else "no",
                        })
                        collected_top += 1
                        collected_check_run_top += 1

                        # ---- Expand GitHub Actions run -> jobs -> steps (if possible) ----
                        if is_gha:
                            check_suite = cr.get("check_suite") or {}
                            cs_id = check_suite.get("id")
                            if isinstance(cs_id, int):
                                if cs_id not in run_cache_by_check_suite:
                                    run_obj = get_workflow_run_by_check_suite_id(gh, full_name, cs_id)
                                    if isinstance(run_obj, dict):
                                        run_cache_by_check_suite[cs_id] = run_obj
                                run_obj = run_cache_by_check_suite.get(cs_id)

                                if isinstance(run_obj, dict):
                                    run_id = run_obj.get("id")
                                    attempt = run_obj.get("run_attempt") or 1

                                    # write run row
                                    rkey = f"gharun:{run_id}:{attempt}"
                                    if rkey not in existing_keys and isinstance(run_id, int):
                                        existing_keys.add(rkey)
                                        raw_rows.append({
                                            "full_name": full_name,
                                            "default_branch": branch,
                                            "episode_id": ep_id,
                                            "episode_env_styles": ep["episode_env_styles"],
                                            "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                                            "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                                            "commit_sha": sha,

                                            "record_type": "gha_run",
                                            "level": "run",
                                            "provider": "github-actions",
                                            "provider_kind": "github_actions",
                                            "is_github_actions": "yes",

                                            "workflow_run_id": run_id,
                                            "workflow_run_attempt": attempt,
                                            "workflow_name": run_obj.get("name") or "",
                                            "workflow_event": run_obj.get("event") or "",
                                            "head_branch": run_obj.get("head_branch") or "",
                                            "head_sha": run_obj.get("head_sha") or "",
                                            "run_created_at": run_obj.get("created_at") or "",
                                            "run_updated_at": run_obj.get("updated_at") or "",

                                            "job_status": run_obj.get("status") or "",
                                            "job_conclusion": run_obj.get("conclusion") or "",

                                            "html_url": run_obj.get("html_url") or "",
                                            "details_url": "",
                                            "collected_at_utc": now_utc_iso(),
                                            "run_instance_key": rkey,
                                        })

                                    # fetch jobs (cached)
                                    if isinstance(run_id, int):
                                        if run_id not in jobs_cache_by_run:
                                            jobs_cache_by_run[run_id] = list_jobs_for_run(gh, full_name, run_id)
                                        jobs = jobs_cache_by_run.get(run_id, [])

                                        for j in jobs:
                                            j_id = j.get("id")
                                            j_name = j.get("name") or ""
                                            steps = j.get("steps") or []
                                            job_match = is_instru(j_name) or any(is_instru((s.get("name") or "")) for s in steps)

                                            if not job_match:
                                                continue

                                            jkey = f"ghajob:{j_id}"
                                            if jkey not in existing_keys and isinstance(j_id, int):
                                                existing_keys.add(jkey)
                                                raw_rows.append({
                                                    "full_name": full_name,
                                                    "default_branch": branch,
                                                    "episode_id": ep_id,
                                                    "episode_env_styles": ep["episode_env_styles"],
                                                    "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                                                    "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                                                    "commit_sha": sha,

                                                    "record_type": "gha_job",
                                                    "level": "job",
                                                    "provider": "github-actions",
                                                    "provider_kind": "github_actions",
                                                    "is_github_actions": "yes",

                                                    "workflow_run_id": run_id,
                                                    "workflow_run_attempt": attempt,

                                                    "job_id": j_id,
                                                    "job_name": j_name,
                                                    "job_status": j.get("status") or "",
                                                    "job_conclusion": j.get("conclusion") or "",
                                                    "runner_name": j.get("runner_name") or "",
                                                    "runner_labels": ",".join(j.get("labels") or []) if isinstance(j.get("labels"), list) else "",

                                                    "started_at": j.get("started_at") or "",
                                                    "completed_at": j.get("completed_at") or "",
                                                    "duration_seconds": seconds_between(j.get("started_at"), j.get("completed_at")) or "",

                                                    "html_url": j.get("html_url") or "",
                                                    "details_url": "",
                                                    "collected_at_utc": now_utc_iso(),
                                                    "run_instance_key": jkey,
                                                })

                                            # steps (optional)
                                            for s in steps:
                                                s_name = s.get("name") or ""
                                                if not is_instru(s_name):
                                                    continue

                                                s_num = s.get("number") or ""
                                                skey = f"ghastep:{j_id}:{s_num}:{s_name}"
                                                if skey in existing_keys:
                                                    continue
                                                existing_keys.add(skey)

                                                raw_rows.append({
                                                    "full_name": full_name,
                                                    "default_branch": branch,
                                                    "episode_id": ep_id,
                                                    "episode_env_styles": ep["episode_env_styles"],
                                                    "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                                                    "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                                                    "commit_sha": sha,

                                                    "record_type": "gha_step",
                                                    "level": "step",
                                                    "provider": "github-actions",
                                                    "provider_kind": "github_actions",
                                                    "is_github_actions": "yes",

                                                    "workflow_run_id": run_id,
                                                    "workflow_run_attempt": attempt,

                                                    "job_id": j_id,
                                                    "job_name": j_name,

                                                    "step_number": s_num,
                                                    "step_name": s_name,
                                                    "step_status": s.get("status") or "",
                                                    "step_conclusion": s.get("conclusion") or "",

                                                    "started_at": s.get("started_at") or "",
                                                    "completed_at": s.get("completed_at") or "",
                                                    "duration_seconds": seconds_between(s.get("started_at"), s.get("completed_at")) or "",

                                                    "html_url": "",
                                                    "details_url": "",
                                                    "collected_at_utc": now_utc_iso(),
                                                    "run_instance_key": skey,
                                                })

                        if collected_top >= K_INSTRU_RECORDS_PER_EPISODE:
                            break

                    if collected_top >= K_INSTRU_RECORDS_PER_EPISODE:
                        break

                    # ---- Commit statuses (external CI often shows up here) ----
                    if allow_commit_status():
                        st0 = get_combined_status_for_commit(gh, full_name, sha)
                        statuses = st0.get("statuses", []) if isinstance(st0, dict) else []
                        for s in statuses:
                            statuses_seen_total += 1

                            context = (s.get("context") or "").strip()
                            if not is_instru(context):
                                continue

                            statuses_instru_context += 1

                            state = (s.get("state") or "").strip().lower()
                            created_at = s.get("created_at") or ""
                            updated_at = s.get("updated_at") or ""
                            dur = seconds_between(created_at, updated_at)

                            strict_pass = True
                            if STRICT_EPISODE_TIME_FILTER:
                                strict_pass = time_filter_pass(created_at, updated_at, start_dt.to_pydatetime(), end_dt.to_pydatetime())

                            if strict_pass:
                                statuses_timefilter_pass += 1

                            if STRICT_EPISODE_TIME_FILTER and not strict_pass:
                                continue

                            prov = (context.split("/")[0] if "/" in context else context.split(":")[0]).strip().lower()

                            skey = f"status:{sha}:{context}:{created_at}"
                            if skey in existing_keys:
                                continue
                            existing_keys.add(skey)

                            raw_rows.append({
                                "full_name": full_name,
                                "default_branch": branch,
                                "episode_id": ep_id,
                                "episode_env_styles": ep["episode_env_styles"],
                                "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                                "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                                "commit_sha": sha,

                                "record_type": "commit_status",
                                "level": "commit_status",
                                "provider": prov or "not_github_actions",
                                "provider_kind": "github_commit_status",
                                "is_github_actions": "no",

                                "job_name": context,
                                "job_conclusion": state,

                                "started_at": created_at,
                                "completed_at": updated_at,
                                "duration_seconds": dur if dur is not None else "",

                                "html_url": s.get("target_url") or "",
                                "details_url": "",
                                "collected_at_utc": now_utc_iso(),
                                "run_instance_key": skey,

                                "started_in_episode": in_episode_window(created_at, start_dt.to_pydatetime(), end_dt.to_pydatetime()),
                                "completed_in_episode": in_episode_window(updated_at, start_dt.to_pydatetime(), end_dt.to_pydatetime()),
                                "strict_time_filter_pass": "yes" if strict_pass else "no",
                            })
                            collected_top += 1
                            collected_commit_status_top += 1
                            if collected_top >= K_INSTRU_RECORDS_PER_EPISODE:
                                break

                # flush window chunk
                flush_rows(raw_rows, RAW_OUT)
                raw_rows.clear()

                cursor = window_end
                window_days = min(MAX_WINDOW_DAYS, max(window_days * 2, window_days + 1))
                windows_advanced += 1

                # checkpoint
                cp.setdefault("episode_state", {})[k] = {
                    "cursor": cursor.isoformat(),
                    "window_days": window_days,
                    "commits_scanned": commits_scanned,
                    "collected_top": collected_top,
                    "windows_advanced": windows_advanced,
                    "checkruns_seen_total": checkruns_seen_total,
                    "checkruns_instru_name": checkruns_instru_name,
                    "checkruns_timefilter_pass": checkruns_timefilter_pass,
                    "statuses_seen_total": statuses_seen_total,
                    "statuses_instru_context": statuses_instru_context,
                    "statuses_timefilter_pass": statuses_timefilter_pass,
                    "collected_check_run_top": collected_check_run_top,
                    "collected_commit_status_top": collected_commit_status_top,
                }
                save_checkpoint(cp)

            print(
                f"[episode] {full_name} ep={ep_id} style={ep['episode_env_styles']} "
                f"collected_top={collected_top} commits_scanned={commits_scanned}"
            )

            # write episode coverage row (helps explain missing episodes)
            append_episode_coverage({
                "full_name": full_name,
                "episode_id": ep_id,
                "episode_env_styles": ep["episode_env_styles"],
                "episode_start_utc": start_dt.isoformat().replace("+00:00", "Z"),
                "episode_end_utc": end_dt.isoformat().replace("+00:00", "Z"),
                "commits_scanned": commits_scanned,
                "windows_advanced": windows_advanced,
                "checkruns_seen_total": checkruns_seen_total,
                "checkruns_instru_name": checkruns_instru_name,
                "checkruns_timefilter_pass": checkruns_timefilter_pass,
                "statuses_seen_total": statuses_seen_total,
                "statuses_instru_context": statuses_instru_context,
                "statuses_timefilter_pass": statuses_timefilter_pass,
                "collected_top_final": collected_top,
                "collected_check_run_top": collected_check_run_top,
                "collected_commit_status_top": collected_commit_status_top,
            })

            # mark done + recompute aggregates so files appear during long runs
            cp.setdefault("episode_done", {})[k] = True
            cp.get("episode_state", {}).pop(k, None)
            save_checkpoint(cp)

            recompute_aggregates()
            print(f"[save] aggregates updated: {WF_EP_OUT.name}, {STYLE_OUT.name}, "
                  f"{WF_EP_OUT_EXEC.name}, {STYLE_OUT_EXEC.name}, {EP_COV_OUT.name}")

    # final aggregates
    recompute_aggregates()
    print("Done.")

if __name__ == "__main__":
    main()
